In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ATML/PA1"
)

REPO_ROOT = Path(
    "/content/drive/MyDrive/ATML/PA1-repo"
)

TASK3_WORK_ROOT = (
    PROJECT_ROOT / "task3"
)

TASK3_REPO_ROOT = (
    REPO_ROOT / "task3"
)

PACS_ROOT = (
    PROJECT_ROOT / "datasets/PACS"
)

SOURCE_SPLIT_PATH = (
    REPO_ROOT
    / "shared/splits/"
    "pacs_sketch_seed6304.json"
)

ERM_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "task2/checkpoints/"
    "source_only_best.pt"
)

print(
    "Repository:",
    REPO_ROOT,
)

print(
    "PACS exists:",
    PACS_ROOT.exists(),
)

print(
    "Shared split exists:",
    SOURCE_SPLIT_PATH.exists(),
)

print(
    "ERM checkpoint exists:",
    ERM_CHECKPOINT_PATH.exists(),
)

Repository: /content/drive/MyDrive/ATML/PA1-repo
PACS exists: True
Shared split exists: True
ERM checkpoint exists: True


In [3]:
repository_folders = [
    "task3/configs",
    "task3/models",
    "task3/methods",
    "task3/evaluation",
    "task3/selection",
    "task3/notebooks",
    "task3/results",
    "task3/figures",
    "task3/scripts",
]

work_folders = [
    "task3/checkpoints",
    "task3/notebooks",
]

for relative_folder in (
    repository_folders
):
    folder_path = (
        REPO_ROOT / relative_folder
    )

    folder_path.mkdir(
        parents=True,
        exist_ok=True,
    )

for relative_folder in work_folders:
    folder_path = (
        PROJECT_ROOT / relative_folder
    )

    folder_path.mkdir(
        parents=True,
        exist_ok=True,
    )

print(
    "Task 3 folders created."
)

Task 3 folders created.


In [4]:
package_directories = [
    TASK3_REPO_ROOT,
    TASK3_REPO_ROOT / "models",
    TASK3_REPO_ROOT / "methods",
    TASK3_REPO_ROOT / "evaluation",
    TASK3_REPO_ROOT / "selection",
]

for package_directory in (
    package_directories
):
    init_path = (
        package_directory
        / "__init__.py"
    )

    init_path.write_text(
        "",
        encoding="utf-8",
    )

    print(
        "Created:",
        init_path.relative_to(
            REPO_ROOT
        ),
    )

Created: task3/__init__.py
Created: task3/models/__init__.py
Created: task3/methods/__init__.py
Created: task3/evaluation/__init__.py
Created: task3/selection/__init__.py


In [5]:
import json


with SOURCE_SPLIT_PATH.open(
    "r",
    encoding="utf-8",
) as split_file:
    pacs_protocol = json.load(
        split_file
    )

print(
    "Seed:",
    pacs_protocol["seed"],
)

print(
    "Source domains:",
    pacs_protocol[
        "source_domains"
    ],
)

print(
    "Target domain reserved:",
    pacs_protocol[
        "target_domain"
    ],
)

print()
print("Source splits:")

for domain_name, split_data in (
    pacs_protocol[
        "source_splits"
    ].items()
):
    print(
        domain_name,
        "| keys:",
        list(split_data.keys()),
    )

    for split_name, indices in (
        split_data.items()
    ):
        print(
            " ",
            split_name,
            len(indices),
        )

Seed: 6304
Source domains: ['photo', 'art_painting', 'cartoon']
Target domain reserved: sketch

Source splits:
photo | keys: ['train_indices', 'validation_indices']
  train_indices 1336
  validation_indices 334
art_painting | keys: ['train_indices', 'validation_indices']
  train_indices 1638
  validation_indices 410
cartoon | keys: ['train_indices', 'validation_indices']
  train_indices 1875
  validation_indices 469


In [6]:
import yaml


task2_base_path = (
    REPO_ROOT
    / "task2/configs/base.yaml"
)

task2_source_path = (
    REPO_ROOT
    / "task2/configs/"
    "source_only.yaml"
)

with task2_base_path.open(
    "r",
    encoding="utf-8",
) as config_file:
    task2_base_config = (
        yaml.safe_load(
            config_file
        )
    )

with task2_source_path.open(
    "r",
    encoding="utf-8",
) as config_file:
    task2_source_config = (
        yaml.safe_load(
            config_file
        )
    )

print("Task 2 base configuration:")
print(
    yaml.safe_dump(
        task2_base_config,
        sort_keys=False,
    )
)

print(
    "Task 2 source-only configuration:"
)

print(
    yaml.safe_dump(
        task2_source_config,
        sort_keys=False,
    )
)

Task 2 base configuration:
seed: 6304
dataset:
  name: PACS
  huggingface_id: flwrlabs/pacs
  source_domains:
  - photo
  - art_painting
  - cartoon
  target_domain: sketch
  number_of_classes: 7
  validation_fraction: 0.2
preprocessing:
  resize:
  - 256
  - 256
  crop_size: 224
  horizontal_flip: true
  normalization: imagenet
model:
  architecture: resnet18
  weights: IMAGENET1K_V1
  feature_dimension: 512
  freeze_batchnorm_running_statistics: true
  train_batchnorm_affine_parameters: true
batching:
  source_examples_per_domain: 8
  total_source_batch_size: 24
  target_batch_size: 24
  evaluation_batch_size: 64
  number_of_workers: 2
optimization:
  optimizer: AdamW
  learning_rate: 0.0001
  weight_decay: 0.0001
  maximum_epochs: 30
  early_stopping_patience: 5
  gradient_clip_norm: 1.0
selection:
  metric: mean_source_validation_macro_f1
  use_target_labels: false

Task 2 source-only configuration:
method:
  name: source_only
  uses_unlabeled_target: false
  classification_loss_we

In [7]:
data_package_path = (
    TASK3_REPO_ROOT / "data"
)

data_package_path.mkdir(
    parents=True,
    exist_ok=True,
)

(
    data_package_path / "__init__.py"
).write_text(
    "",
    encoding="utf-8",
)


task3_base_config = {
    "seed": 6304,
    "dataset": {
        "name": "PACS",
        "source_domains": [
            "photo",
            "art_painting",
            "cartoon",
        ],
        "reserved_target_domain": (
            "sketch"
        ),
        "number_of_classes": 7,
        "source_split_path": (
            "shared/splits/"
            "pacs_sketch_seed6304.json"
        ),
    },
    "data_policy": {
        "load_target_during_training": (
            False
        ),
        "load_target_during_selection": (
            False
        ),
        "load_target_during_diagnostics": (
            False
        ),
        "target_access_stage": (
            "final_evaluation_only"
        ),
    },
    "model": {
        "architecture": "resnet18",
        "weights": "IMAGENET1K_V1",
        "feature_dimension": 512,
        "number_of_classes": 7,
        "freeze_batchnorm_running_statistics": (
            True
        ),
        "train_batchnorm_affine_parameters": (
            True
        ),
    },
    "batching": {
        "examples_per_source_domain": 8,
        "total_source_batch_size": 24,
        "evaluation_batch_size": 64,
        "number_of_workers": 2,
    },
    "optimization": {
        "optimizer": "AdamW",
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
        "maximum_epochs": 30,
        "early_stopping_patience": 5,
        "gradient_clip_norm": 1.0,
    },
    "selection": {
        "metric": (
            "mean_source_validation_"
            "macro_f1"
        ),
        "use_target_images": False,
        "use_target_labels": False,
    },
}

erm_config = {
    "method": {
        "name": "erm",
        "training_action": (
            "reuse_task2_source_only"
        ),
        "retrain": False,
    },
    "checkpoint": {
        "external_filename": (
            "source_only_best.pt"
        ),
    },
}

dan_dg_config = {
    "method": {
        "name": "dan_dg",
        "classification_loss_weight": (
            1.0
        ),
        "mmd_weight": 1.0,
        "alignment": (
            "mean_pairwise_source_mmd"
        ),
        "source_domain_pairs": [
            ["photo", "art_painting"],
            ["photo", "cartoon"],
            ["art_painting", "cartoon"],
        ],
        "kernel": "multi_rbf",
        "bandwidth_multipliers": [
            0.5,
            1.0,
            2.0,
        ],
        "bandwidth_statistic": (
            "median_pairwise_"
            "squared_distance"
        ),
    },
    "checkpoint": {
        "filename": "dan_dg_best.pt",
    },
}

sam_config = {
    "method": {
        "name": "sam",
        "rho": 0.05,
        "adaptive": False,
        "classification_loss_weight": (
            1.0
        ),
        "two_forward_backward_passes": (
            True
        ),
    },
    "checkpoint": {
        "filename": "sam_best.pt",
    },
}

controlled_study_config = {
    "study": {
        "method": "dan_dg",
        "parameter": "mmd_weight",
        "values": [
            0.1,
            1.0,
            10.0,
        ],
        "main_comparison_value": 1.0,
        "selection_data": (
            "source_validation_only"
        ),
        "target_results_used_to_choose_study": (
            False
        ),
    },
    "preregistered_expectation": {
        "source_domain_separability": (
            "Expected to decrease as "
            "MMD weight increases."
        ),
        "source_classification": (
            "Excessive alignment may "
            "remove class information."
        ),
        "unseen_target_performance": (
            "Not assumed to improve "
            "monotonically with alignment."
        ),
    },
}

configuration_files = {
    "base.yaml": task3_base_config,
    "erm.yaml": erm_config,
    "dan_dg.yaml": dan_dg_config,
    "sam.yaml": sam_config,
    "controlled_study.yaml": (
        controlled_study_config
    ),
}

for file_name, configuration in (
    configuration_files.items()
):
    configuration_path = (
        TASK3_REPO_ROOT
        / "configs"
        / file_name
    )

    with configuration_path.open(
        "w",
        encoding="utf-8",
    ) as config_file:
        yaml.safe_dump(
            configuration,
            config_file,
            sort_keys=False,
        )

    print(
        "Created:",
        configuration_path.relative_to(
            REPO_ROOT
        ),
    )

Created: task3/configs/base.yaml
Created: task3/configs/erm.yaml
Created: task3/configs/dan_dg.yaml
Created: task3/configs/sam.yaml
Created: task3/configs/controlled_study.yaml


In [9]:
%cd /content/drive/MyDrive/ATML/PA1-repo

/content/drive/MyDrive/ATML/PA1-repo


In [10]:
%%writefile task3/data/source_loaders.py
import random

import numpy as np
import torch

from torch.utils.data import DataLoader

from shared.pacs import (
    PACSLabeledDataset,
    build_pacs_evaluation_transform,
    build_pacs_train_transform,
)


def _seed_worker(worker_id):
    worker_seed = (
        torch.initial_seed()
        % (2 ** 32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def build_task3_source_datasets(
    dataset,
    protocol,
):
    source_domains = protocol[
        "source_domains"
    ]

    reserved_target = protocol[
        "target_domain"
    ]

    if reserved_target in source_domains:
        raise ValueError(
            "The reserved target domain "
            "cannot be a source domain."
        )

    train_transform = (
        build_pacs_train_transform()
    )

    evaluation_transform = (
        build_pacs_evaluation_transform()
    )

    source_train = {}
    source_validation = {}

    for domain_name in source_domains:
        domain_split = protocol[
            "source_splits"
        ][domain_name]

        source_train[domain_name] = (
            PACSLabeledDataset(
                dataset=dataset,
                indices=domain_split[
                    "train_indices"
                ],
                transform=train_transform,
                domain_name=domain_name,
            )
        )

        source_validation[
            domain_name
        ] = PACSLabeledDataset(
            dataset=dataset,
            indices=domain_split[
                "validation_indices"
            ],
            transform=(
                evaluation_transform
            ),
            domain_name=domain_name,
        )

    return {
        "source_train": source_train,
        "source_validation": (
            source_validation
        ),
    }


def build_task3_source_loaders(
    datasets,
    source_batch_size=8,
    evaluation_batch_size=64,
    number_of_workers=2,
    seed=6304,
):
    source_train_loaders = {}
    source_validation_loaders = {}

    for domain_index, (
        domain_name,
        domain_dataset,
    ) in enumerate(
        datasets[
            "source_train"
        ].items()
    ):
        generator = torch.Generator()

        generator.manual_seed(
            seed + domain_index
        )

        source_train_loaders[
            domain_name
        ] = DataLoader(
            domain_dataset,
            batch_size=source_batch_size,
            shuffle=True,
            drop_last=True,
            num_workers=(
                number_of_workers
            ),
            pin_memory=True,
            worker_init_fn=_seed_worker,
            generator=generator,
        )

    for domain_name, domain_dataset in (
        datasets[
            "source_validation"
        ].items()
    ):
        source_validation_loaders[
            domain_name
        ] = DataLoader(
            domain_dataset,
            batch_size=(
                evaluation_batch_size
            ),
            shuffle=False,
            drop_last=False,
            num_workers=(
                number_of_workers
            ),
            pin_memory=True,
            worker_init_fn=_seed_worker,
        )

    steps_per_epoch = max(
        len(loader)
        for loader in (
            source_train_loaders.values()
        )
    )

    return {
        "source_train": (
            source_train_loaders
        ),
        "source_validation": (
            source_validation_loaders
        ),
        "steps_per_epoch": (
            steps_per_epoch
        ),
    }


def cycle_loader(data_loader):
    while True:
        for batch in data_loader:
            yield batch

Writing task3/data/source_loaders.py


In [11]:
from datasets import load_from_disk

from shared.pacs_protocol import (
    load_pacs_protocol,
)

from task3.data.source_loaders import (
    build_task3_source_datasets,
    build_task3_source_loaders,
)


pacs_dataset = load_from_disk(
    str(PACS_ROOT)
)

pacs_protocol = load_pacs_protocol(
    SOURCE_SPLIT_PATH
)

task3_datasets = (
    build_task3_source_datasets(
        dataset=pacs_dataset,
        protocol=pacs_protocol,
    )
)

task3_loaders = (
    build_task3_source_loaders(
        datasets=task3_datasets,
        source_batch_size=8,
        evaluation_batch_size=64,
        number_of_workers=0,
        seed=6304,
    )
)

print(
    "Dataset groups:",
    list(task3_datasets.keys()),
)

print(
    "Loader groups:",
    list(task3_loaders.keys()),
)

print(
    "Training domains:",
    list(
        task3_loaders[
            "source_train"
        ].keys()
    ),
)

print(
    "Validation domains:",
    list(
        task3_loaders[
            "source_validation"
        ].keys()
    ),
)

print(
    "Steps per epoch:",
    task3_loaders[
        "steps_per_epoch"
    ],
)

for domain_name, data_loader in (
    task3_loaders[
        "source_train"
    ].items()
):
    batch = next(iter(data_loader))

    print(
        domain_name,
        "| batch size:",
        len(batch["label"]),
        "| image shape:",
        tuple(batch["image"].shape),
    )

Dataset groups: ['source_train', 'source_validation']
Loader groups: ['source_train', 'source_validation', 'steps_per_epoch']
Training domains: ['photo', 'art_painting', 'cartoon']
Validation domains: ['photo', 'art_painting', 'cartoon']
Steps per epoch: 234


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


photo | batch size: 8 | image shape: (8, 3, 224, 224)
art_painting | batch size: 8 | image shape: (8, 3, 224, 224)
cartoon | batch size: 8 | image shape: (8, 3, 224, 224)


In [12]:
%cd /content/drive/MyDrive/ATML/PA1-repo

!python -m py_compile task3/data/source_loaders.py

print("Task 3 source loader compiled.")

!git status --short

/content/drive/MyDrive/ATML/PA1-repo
Task 3 source loader compiled.
Refresh index: 100% (126/126), done.
?? task3/


In [13]:
import inspect

import task2.methods.dan as task2_dan


print("Available DAN functions:")

for function_name in dir(task2_dan):
    if not function_name.startswith("_"):
        candidate = getattr(
            task2_dan,
            function_name,
        )

        if inspect.isfunction(candidate):
            print(
                function_name,
                inspect.signature(
                    candidate
                ),
            )

print()
print("Task 2 DAN implementation:")
print(
    inspect.getsource(
        task2_dan
    )
)

Available DAN functions:
compute_dan_loss (model, source_images, source_labels, target_images, mmd_weight=1.0)
median_squared_distance (combined_features, epsilon=1e-08)
multi_kernel_mmd (source_features, target_features)
multi_kernel_rbf (first_features, second_features, bandwidths)
pairwise_squared_distance (first_features, second_features)

Task 2 DAN implementation:

import torch
import torch.nn.functional as F


def pairwise_squared_distance(
    first_features,
    second_features,
):
    return torch.cdist(
        first_features,
        second_features,
        p=2,
    ).pow(2)


def median_squared_distance(
    combined_features,
    epsilon=1e-8,
):
    distance_matrix = (
        pairwise_squared_distance(
            combined_features,
            combined_features,
        )
    )

    number_of_samples = (
        distance_matrix.shape[0]
    )

    off_diagonal_mask = (
        ~torch.eye(
            number_of_samples,
            dtype=torch.bool,
            device=

In [14]:
%%writefile task3/methods/erm.py
import torch
import torch.nn.functional as F


def compute_erm_loss(
    model,
    images_by_domain,
    labels_by_domain,
):
    domain_names = list(
        images_by_domain.keys()
    )

    if set(domain_names) != set(
        labels_by_domain.keys()
    ):
        raise ValueError(
            "Image and label domains "
            "do not match."
        )

    losses_by_domain = {}
    logits_by_domain = {}
    features_by_domain = {}

    for domain_name in domain_names:
        logits, features = model(
            images_by_domain[
                domain_name
            ],
            return_features=True,
        )

        domain_loss = F.cross_entropy(
            logits,
            labels_by_domain[
                domain_name
            ],
        )

        losses_by_domain[
            domain_name
        ] = domain_loss

        logits_by_domain[
            domain_name
        ] = logits

        features_by_domain[
            domain_name
        ] = features

    classification_loss = (
        torch.stack(
            list(
                losses_by_domain.values()
            )
        ).mean()
    )

    return {
        "total_loss": (
            classification_loss
        ),
        "classification_loss": (
            classification_loss
        ),
        "alignment_loss": None,
        "losses_by_domain": (
            losses_by_domain
        ),
        "logits_by_domain": (
            logits_by_domain
        ),
        "features_by_domain": (
            features_by_domain
        ),
    }

Writing task3/methods/erm.py


In [15]:
%%writefile task3/methods/dan_dg.py
from itertools import combinations

import torch

from task2.methods.dan import (
    multi_kernel_mmd,
)

from task3.methods.erm import (
    compute_erm_loss,
)


def compute_dan_dg_loss(
    model,
    images_by_domain,
    labels_by_domain,
    mmd_weight=1.0,
):
    erm_outputs = compute_erm_loss(
        model=model,
        images_by_domain=(
            images_by_domain
        ),
        labels_by_domain=(
            labels_by_domain
        ),
    )

    features_by_domain = (
        erm_outputs[
            "features_by_domain"
        ]
    )

    domain_names = list(
        features_by_domain.keys()
    )

    pairwise_mmd_losses = {}
    pairwise_median_distances = {}

    for first_domain, second_domain in (
        combinations(
            domain_names,
            2,
        )
    ):
        mmd_loss, median_distance = (
            multi_kernel_mmd(
                features_by_domain[
                    first_domain
                ],
                features_by_domain[
                    second_domain
                ],
            )
        )

        pair_name = (
            f"{first_domain}__"
            f"{second_domain}"
        )

        pairwise_mmd_losses[
            pair_name
        ] = mmd_loss

        pairwise_median_distances[
            pair_name
        ] = median_distance

    alignment_loss = torch.stack(
        list(
            pairwise_mmd_losses.values()
        )
    ).mean()

    total_loss = (
        erm_outputs[
            "classification_loss"
        ]
        + float(mmd_weight)
        * alignment_loss
    )

    return {
        "total_loss": total_loss,
        "classification_loss": (
            erm_outputs[
                "classification_loss"
            ]
        ),
        "alignment_loss": (
            alignment_loss
        ),
        "losses_by_domain": (
            erm_outputs[
                "losses_by_domain"
            ]
        ),
        "logits_by_domain": (
            erm_outputs[
                "logits_by_domain"
            ]
        ),
        "features_by_domain": (
            features_by_domain
        ),
        "pairwise_mmd_losses": (
            pairwise_mmd_losses
        ),
        "pairwise_median_distances": (
            pairwise_median_distances
        ),
        "mmd_weight": float(
            mmd_weight
        ),
    }

Writing task3/methods/dan_dg.py


In [16]:
%cd /content/drive/MyDrive/ATML/PA1-repo

!python -m py_compile task3/methods/erm.py
!python -m py_compile task3/methods/dan_dg.py

print(
    "ERM and DAN-DG objectives compiled."
)

/content/drive/MyDrive/ATML/PA1-repo
ERM and DAN-DG objectives compiled.


In [17]:
%%writefile task3/methods/sam.py
import torch


class SAMOptimizer:
    def __init__(
        self,
        base_optimizer,
        rho=0.05,
        adaptive=False,
        epsilon=1e-12,
    ):
        if rho < 0:
            raise ValueError(
                "rho must be non-negative."
            )

        self.base_optimizer = (
            base_optimizer
        )

        self.rho = float(rho)
        self.adaptive = bool(adaptive)
        self.epsilon = float(epsilon)

        self._perturbations = {}

    @property
    def param_groups(self):
        return (
            self.base_optimizer.param_groups
        )

    def zero_grad(
        self,
        set_to_none=True,
    ):
        self.base_optimizer.zero_grad(
            set_to_none=set_to_none
        )

    def _parameters_with_gradients(
        self,
    ):
        return [
            parameter
            for parameter_group
            in self.param_groups
            for parameter
            in parameter_group["params"]
            if parameter.grad is not None
        ]

    def gradient_norm(self):
        parameters = (
            self._parameters_with_gradients()
        )

        if not parameters:
            raise RuntimeError(
                "SAM received no gradients."
            )

        gradient_norms = []

        for parameter in parameters:
            if self.adaptive:
                scaled_gradient = (
                    parameter.detach().abs()
                    * parameter.grad
                )
            else:
                scaled_gradient = (
                    parameter.grad
                )

            gradient_norms.append(
                scaled_gradient.norm(
                    p=2
                )
            )

        return torch.stack(
            gradient_norms
        ).norm(
            p=2
        )

    @torch.no_grad()
    def first_step(self):
        gradient_norm = (
            self.gradient_norm()
        )

        scale = (
            self.rho
            / (
                gradient_norm
                + self.epsilon
            )
        )

        self._perturbations = {}

        for parameter_group in (
            self.param_groups
        ):
            for parameter in (
                parameter_group["params"]
            ):
                if parameter.grad is None:
                    continue

                if self.adaptive:
                    parameter_scale = (
                        parameter.detach()
                        .pow(2)
                    )
                else:
                    parameter_scale = 1.0

                perturbation = (
                    parameter_scale
                    * parameter.grad
                    * scale
                )

                parameter.add_(
                    perturbation
                )

                self._perturbations[
                    parameter
                ] = perturbation

        return gradient_norm.detach()

    @torch.no_grad()
    def restore_parameters(self):
        for parameter, perturbation in (
            self._perturbations.items()
        ):
            parameter.sub_(
                perturbation
            )

        self._perturbations = {}

    @torch.no_grad()
    def second_step(self):
        self.restore_parameters()

        self.base_optimizer.step()

    def state_dict(self):
        return {
            "base_optimizer": (
                self.base_optimizer
                .state_dict()
            ),
            "rho": self.rho,
            "adaptive": self.adaptive,
            "epsilon": self.epsilon,
        }

    def load_state_dict(
        self,
        state_dict,
    ):
        self.base_optimizer.load_state_dict(
            state_dict[
                "base_optimizer"
            ]
        )

        self.rho = float(
            state_dict["rho"]
        )

        self.adaptive = bool(
            state_dict["adaptive"]
        )

        self.epsilon = float(
            state_dict["epsilon"]
        )

Writing task3/methods/sam.py


In [18]:
import torch
import torch.nn.functional as F

from task3.methods.sam import (
    SAMOptimizer,
)


torch.manual_seed(6304)

test_model = torch.nn.Linear(
    4,
    3,
)

base_optimizer = torch.optim.AdamW(
    test_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

sam_optimizer = SAMOptimizer(
    base_optimizer=base_optimizer,
    rho=0.05,
    adaptive=False,
)

test_inputs = torch.randn(
    8,
    4,
)

test_labels = torch.randint(
    low=0,
    high=3,
    size=(8,),
)

original_parameters = [
    parameter.detach().clone()
    for parameter in (
        test_model.parameters()
    )
]

first_logits = test_model(
    test_inputs
)

first_loss = F.cross_entropy(
    first_logits,
    test_labels,
)

first_loss.backward()

gradient_norm = (
    sam_optimizer.first_step()
)

perturbed_parameters = [
    parameter.detach().clone()
    for parameter in (
        test_model.parameters()
    )
]

sam_optimizer.zero_grad()

second_logits = test_model(
    test_inputs
)

second_loss = F.cross_entropy(
    second_logits,
    test_labels,
)

second_loss.backward()

sam_optimizer.second_step()

updated_parameters = [
    parameter.detach().clone()
    for parameter in (
        test_model.parameters()
    )
]

was_perturbed = any(
    not torch.equal(
        original,
        perturbed,
    )
    for original, perturbed in zip(
        original_parameters,
        perturbed_parameters,
    )
)

was_updated = any(
    not torch.equal(
        original,
        updated,
    )
    for original, updated in zip(
        original_parameters,
        updated_parameters,
    )
)

print(
    "First loss:",
    first_loss.item(),
)

print(
    "Perturbed loss:",
    second_loss.item(),
)

print(
    "Gradient norm:",
    gradient_norm.item(),
)

print(
    "Parameters were perturbed:",
    was_perturbed,
)

print(
    "Parameters were updated:",
    was_updated,
)

First loss: 1.3922216892242432
Perturbed loss: 1.425567388534546
Gradient norm: 0.6600349545478821
Parameters were perturbed: True
Parameters were updated: True


In [19]:
%cd /content/drive/MyDrive/ATML/PA1-repo

!python -m py_compile task3/methods/sam.py

print("SAM implementation compiled.")

/content/drive/MyDrive/ATML/PA1-repo
SAM implementation compiled.


In [20]:
import inspect

from task2.evaluation.source_validation import (
    evaluate_source_domains,
)

from task2.models.backbone import (
    freeze_batchnorm_statistics,
)

from task2.selection.checkpointing import (
    SourceValidationSelector,
)


print(
    "SourceValidationSelector:"
)

print(
    inspect.getsource(
        SourceValidationSelector
    )
)

print()
print(
    "evaluate_source_domains signature:"
)

print(
    inspect.signature(
        evaluate_source_domains
    )
)

print()
print(
    "freeze_batchnorm_statistics:"
)

print(
    inspect.getsource(
        freeze_batchnorm_statistics
    )
)

SourceValidationSelector:
class SourceValidationSelector:
    def __init__(
        self,
        checkpoint_path,
        patience=5,
        minimum_improvement=0.0,
    ):
        self.checkpoint_path = Path(
            checkpoint_path
        )

        self.patience = int(patience)

        self.minimum_improvement = float(
            minimum_improvement
        )

        self.best_score = float("-inf")
        self.best_epoch = None
        self.epochs_without_improvement = 0

    def update(
        self,
        score,
        epoch,
        model,
        optimizer,
        additional_models=None,
        extra_state=None,
    ):
        score = float(score)
        epoch = int(epoch)

        improved = (
            score
            > self.best_score
            + self.minimum_improvement
        )

        if improved:
            self.best_score = score
            self.best_epoch = epoch
            self.epochs_without_improvement = 0

            if additional_models

In [21]:
%%writefile task3/train.py
import pandas as pd
import torch

from tqdm.auto import tqdm

from task2.evaluation.source_validation import (
    evaluate_source_domains,
)

from task2.models.backbone import (
    freeze_batchnorm_statistics,
)

from task3.data.source_loaders import (
    cycle_loader,
)

from task3.methods.dan_dg import (
    compute_dan_dg_loss,
)

from task3.methods.erm import (
    compute_erm_loss,
)


SUPPORTED_METHODS = {
    "dan_dg",
    "sam",
}


def _prepare_source_batches(
    source_iterators,
    device,
):
    images_by_domain = {}
    labels_by_domain = {}

    for domain_name, iterator in (
        source_iterators.items()
    ):
        batch = next(iterator)

        images_by_domain[
            domain_name
        ] = batch["image"].to(
            device,
            non_blocking=True,
        )

        labels_by_domain[
            domain_name
        ] = batch["label"].to(
            device,
            non_blocking=True,
        )

    return (
        images_by_domain,
        labels_by_domain,
    )


def _classification_accuracy(
    logits_by_domain,
    labels_by_domain,
):
    number_correct = 0
    number_total = 0

    for domain_name, logits in (
        logits_by_domain.items()
    ):
        predictions = logits.argmax(
            dim=1
        )

        labels = labels_by_domain[
            domain_name
        ]

        number_correct += int(
            (
                predictions == labels
            ).sum().item()
        )

        number_total += int(
            labels.numel()
        )

    return (
        number_correct
        / max(number_total, 1)
    )


def train_task3_method(
    method_name,
    model,
    loaders,
    optimizer,
    selector,
    device,
    maximum_epochs=30,
    mmd_weight=1.0,
    gradient_clip_norm=1.0,
    number_of_classes=7,
):
    if method_name not in (
        SUPPORTED_METHODS
    ):
        raise ValueError(
            f"Unsupported Task 3 method: "
            f"{method_name}"
        )

    source_train_loaders = loaders[
        "source_train"
    ]

    source_validation_loaders = (
        loaders[
            "source_validation"
        ]
    )

    steps_per_epoch = int(
        loaders["steps_per_epoch"]
    )

    training_rows = []

    for epoch in range(
        1,
        maximum_epochs + 1,
    ):
        model.train()

        freeze_batchnorm_statistics(
            model
        )

        source_iterators = {
            domain_name: cycle_loader(
                data_loader
            )
            for domain_name, data_loader
            in source_train_loaders.items()
        }

        epoch_total_loss = 0.0
        epoch_classification_loss = 0.0
        epoch_alignment_loss = 0.0
        epoch_unperturbed_loss = 0.0
        epoch_gradient_norm = 0.0
        epoch_source_accuracy = 0.0

        progress_bar = tqdm(
            range(steps_per_epoch),
            desc=(
                f"{method_name} "
                f"epoch {epoch}"
            ),
        )

        for _ in progress_bar:
            (
                images_by_domain,
                labels_by_domain,
            ) = _prepare_source_batches(
                source_iterators=(
                    source_iterators
                ),
                device=device,
            )

            if method_name == "dan_dg":
                optimizer.zero_grad(
                    set_to_none=True
                )

                outputs = (
                    compute_dan_dg_loss(
                        model=model,
                        images_by_domain=(
                            images_by_domain
                        ),
                        labels_by_domain=(
                            labels_by_domain
                        ),
                        mmd_weight=(
                            mmd_weight
                        ),
                    )
                )

                outputs[
                    "total_loss"
                ].backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=(
                        gradient_clip_norm
                    ),
                )

                optimizer.step()

                total_loss = outputs[
                    "total_loss"
                ]

                classification_loss = (
                    outputs[
                        "classification_loss"
                    ]
                )

                alignment_loss = outputs[
                    "alignment_loss"
                ]

                unperturbed_loss = (
                    classification_loss
                )

                gradient_norm_value = (
                    float("nan")
                )

                accuracy_logits = outputs[
                    "logits_by_domain"
                ]

            else:
                optimizer.zero_grad(
                    set_to_none=True
                )

                first_outputs = (
                    compute_erm_loss(
                        model=model,
                        images_by_domain=(
                            images_by_domain
                        ),
                        labels_by_domain=(
                            labels_by_domain
                        ),
                    )
                )

                first_outputs[
                    "total_loss"
                ].backward()

                sam_gradient_norm = (
                    optimizer.first_step()
                )

                optimizer.zero_grad(
                    set_to_none=True
                )

                try:
                    freeze_batchnorm_statistics(
                        model
                    )

                    second_outputs = (
                        compute_erm_loss(
                            model=model,
                            images_by_domain=(
                                images_by_domain
                            ),
                            labels_by_domain=(
                                labels_by_domain
                            ),
                        )
                    )

                    second_outputs[
                        "total_loss"
                    ].backward()

                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        max_norm=(
                            gradient_clip_norm
                        ),
                    )

                    optimizer.second_step()

                except Exception:
                    optimizer.restore_parameters()
                    raise

                total_loss = second_outputs[
                    "total_loss"
                ]

                classification_loss = (
                    second_outputs[
                        "classification_loss"
                    ]
                )

                alignment_loss = None

                unperturbed_loss = (
                    first_outputs[
                        "classification_loss"
                    ]
                )

                gradient_norm_value = float(
                    sam_gradient_norm.item()
                )

                accuracy_logits = (
                    first_outputs[
                        "logits_by_domain"
                    ]
                )

            batch_accuracy = (
                _classification_accuracy(
                    logits_by_domain=(
                        accuracy_logits
                    ),
                    labels_by_domain=(
                        labels_by_domain
                    ),
                )
            )

            epoch_total_loss += float(
                total_loss.detach().item()
            )

            epoch_classification_loss += (
                float(
                    classification_loss
                    .detach()
                    .item()
                )
            )

            if alignment_loss is not None:
                epoch_alignment_loss += (
                    float(
                        alignment_loss
                        .detach()
                        .item()
                    )
                )

            epoch_unperturbed_loss += float(
                unperturbed_loss
                .detach()
                .item()
            )

            if method_name == "sam":
                epoch_gradient_norm += (
                    gradient_norm_value
                )

            epoch_source_accuracy += (
                batch_accuracy
            )

            progress_bar.set_postfix(
                loss=(
                    epoch_total_loss
                    / (
                        progress_bar.n + 1
                    )
                )
            )

        validation_results = (
            evaluate_source_domains(
                model=model,
                validation_loaders=(
                    source_validation_loaders
                ),
                device=device,
                number_of_classes=(
                    number_of_classes
                ),
            )
        )

        mean_macro_f1 = (
            validation_results[
                "aggregate"
            ]["mean_macro_f1"]
        )

        history_row = {
            "epoch": epoch,
            "training_total_loss": (
                epoch_total_loss
                / steps_per_epoch
            ),
            "training_classification_loss": (
                epoch_classification_loss
                / steps_per_epoch
            ),
            "training_alignment_loss": (
                epoch_alignment_loss
                / steps_per_epoch
                if method_name
                == "dan_dg"
                else float("nan")
            ),
            "training_unperturbed_loss": (
                epoch_unperturbed_loss
                / steps_per_epoch
            ),
            "training_source_accuracy": (
                epoch_source_accuracy
                / steps_per_epoch
            ),
            "mean_sam_gradient_norm": (
                epoch_gradient_norm
                / steps_per_epoch
                if method_name == "sam"
                else float("nan")
            ),
            "mean_source_validation_accuracy": (
                validation_results[
                    "aggregate"
                ]["mean_accuracy"]
            ),
            "mean_source_validation_macro_f1": (
                mean_macro_f1
            ),
            "worst_source_validation_accuracy": (
                validation_results[
                    "aggregate"
                ]["worst_accuracy"]
            ),
            "worst_source_validation_macro_f1": (
                validation_results[
                    "aggregate"
                ]["worst_macro_f1"]
            ),
        }

        for domain_name, metrics in (
            validation_results[
                "domains"
            ].items()
        ):
            history_row[
                f"{domain_name}_"
                "validation_accuracy"
            ] = metrics["accuracy"]

            history_row[
                f"{domain_name}_"
                "validation_macro_f1"
            ] = metrics["macro_f1"]

        training_rows.append(
            history_row
        )

        selector.update(
            score=mean_macro_f1,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            extra_state={
                "method": method_name,
                "mmd_weight": (
                    float(mmd_weight)
                    if method_name
                    == "dan_dg"
                    else None
                ),
            },
        )

        print(
            f"Epoch {epoch}: "
            f"train loss="
            f"{history_row['training_total_loss']:.4f}, "
            f"mean validation macro-F1="
            f"{mean_macro_f1:.4f}, "
            f"best={selector.best_score:.4f}"
        )

        if selector.should_stop:
            print(
                "Early stopping triggered."
            )
            break

    selected_checkpoint = torch.load(
        selector.checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        selected_checkpoint[
            "model_state_dict"
        ]
    )

    training_history = pd.DataFrame(
        training_rows
    )

    return {
        "model": model,
        "history": training_history,
        "selected_checkpoint": (
            selected_checkpoint
        ),
    }

Writing task3/train.py


In [22]:
%cd /content/drive/MyDrive/ATML/PA1-repo

!python -m py_compile task3/train.py

import inspect

from task3.train import (
    train_task3_method,
)

print(
    "Training signature:"
)

print(
    inspect.signature(
        train_task3_method
    )
)

print()
print(
    "Task 3 training loop compiled."
)

/content/drive/MyDrive/ATML/PA1-repo
Training signature:
(method_name, model, loaders, optimizer, selector, device, maximum_epochs=30, mmd_weight=1.0, gradient_clip_norm=1.0, number_of_classes=7)

Task 3 training loop compiled.


In [23]:
from pathlib import Path

import torch

from torch.utils.data import (
    DataLoader,
    Dataset,
)

from common.seed import set_seed

from task2.selection.checkpointing import (
    SourceValidationSelector,
)

from task3.methods.sam import (
    SAMOptimizer,
)

from task3.train import (
    train_task3_method,
)


class TinyDomainDataset(Dataset):
    def __init__(
        self,
        domain_offset,
        number_of_samples=12,
    ):
        generator = torch.Generator()
        generator.manual_seed(
            6304 + int(
                domain_offset * 10
            )
        )

        self.images = (
            torch.randn(
                number_of_samples,
                4,
                generator=generator,
            )
            + domain_offset
        )

        self.labels = torch.randint(
            low=0,
            high=7,
            size=(number_of_samples,),
            generator=generator,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            "image": self.images[index],
            "label": self.labels[index],
        }


class TinyDGModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.feature_extractor = (
            torch.nn.Sequential(
                torch.nn.Linear(
                    4,
                    16,
                ),
                torch.nn.ReLU(),
                torch.nn.Linear(
                    16,
                    8,
                ),
                torch.nn.ReLU(),
            )
        )

        self.classifier = (
            torch.nn.Linear(
                8,
                7,
            )
        )

    def forward(
        self,
        images,
        return_features=False,
    ):
        features = (
            self.feature_extractor(
                images
            )
        )

        logits = self.classifier(
            features
        )

        if return_features:
            return logits, features

        return logits


def build_tiny_loaders():
    domain_offsets = {
        "photo": 0.0,
        "art_painting": 0.5,
        "cartoon": 1.0,
    }

    train_loaders = {}
    validation_loaders = {}

    for domain_name, offset in (
        domain_offsets.items()
    ):
        train_dataset = (
            TinyDomainDataset(
                domain_offset=offset,
                number_of_samples=12,
            )
        )

        validation_dataset = (
            TinyDomainDataset(
                domain_offset=offset,
                number_of_samples=8,
            )
        )

        train_loaders[
            domain_name
        ] = DataLoader(
            train_dataset,
            batch_size=4,
            shuffle=False,
            drop_last=True,
        )

        validation_loaders[
            domain_name
        ] = DataLoader(
            validation_dataset,
            batch_size=4,
            shuffle=False,
        )

    return {
        "source_train": train_loaders,
        "source_validation": (
            validation_loaders
        ),
        "steps_per_epoch": 2,
    }


tiny_loaders = build_tiny_loaders()

smoke_checkpoint_directory = (
    TASK3_WORK_ROOT
    / "checkpoints"
)

smoke_checkpoint_directory.mkdir(
    parents=True,
    exist_ok=True,
)


set_seed(6304)

dan_dg_model = TinyDGModel()

dan_dg_optimizer = (
    torch.optim.AdamW(
        dan_dg_model.parameters(),
        lr=1e-4,
        weight_decay=1e-4,
    )
)

dan_dg_checkpoint = (
    smoke_checkpoint_directory
    / "smoke_dan_dg.pt"
)

dan_dg_selector = (
    SourceValidationSelector(
        checkpoint_path=(
            dan_dg_checkpoint
        ),
        patience=1,
    )
)

dan_dg_result = train_task3_method(
    method_name="dan_dg",
    model=dan_dg_model,
    loaders=tiny_loaders,
    optimizer=dan_dg_optimizer,
    selector=dan_dg_selector,
    device=torch.device("cpu"),
    maximum_epochs=1,
    mmd_weight=1.0,
    gradient_clip_norm=1.0,
    number_of_classes=7,
)


set_seed(6304)

sam_model = TinyDGModel()

sam_base_optimizer = (
    torch.optim.AdamW(
        sam_model.parameters(),
        lr=1e-4,
        weight_decay=1e-4,
    )
)

sam_optimizer = SAMOptimizer(
    base_optimizer=(
        sam_base_optimizer
    ),
    rho=0.05,
    adaptive=False,
)

sam_checkpoint = (
    smoke_checkpoint_directory
    / "smoke_sam.pt"
)

sam_selector = (
    SourceValidationSelector(
        checkpoint_path=(
            sam_checkpoint
        ),
        patience=1,
    )
)

sam_result = train_task3_method(
    method_name="sam",
    model=sam_model,
    loaders=tiny_loaders,
    optimizer=sam_optimizer,
    selector=sam_selector,
    device=torch.device("cpu"),
    maximum_epochs=1,
    gradient_clip_norm=1.0,
    number_of_classes=7,
)


print()
print(
    "DAN-DG history rows:",
    len(
        dan_dg_result["history"]
    ),
)

print(
    "SAM history rows:",
    len(
        sam_result["history"]
    ),
)

print(
    "DAN-DG checkpoint created:",
    dan_dg_checkpoint.exists(),
)

print(
    "SAM checkpoint created:",
    sam_checkpoint.exists(),
)

dan_dg_checkpoint.unlink(
    missing_ok=True
)

sam_checkpoint.unlink(
    missing_ok=True
)

print(
    "Temporary smoke-test "
    "checkpoints removed."
)

dan_dg epoch 1:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1: train loss=3.0581, mean validation macro-F1=0.0714, best=0.0714


sam epoch 1:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1: train loss=1.9617, mean validation macro-F1=0.0714, best=0.0714

DAN-DG history rows: 1
SAM history rows: 1
DAN-DG checkpoint created: True
SAM checkpoint created: True
Temporary smoke-test checkpoints removed.


In [2]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

from pathlib import Path

REPO_ROOT = Path(
    "/content/drive/MyDrive/ATML/PA1-repo"
)

TASK3_REPO_ROOT = (
    REPO_ROOT / "task3"
)

%cd /content/drive/MyDrive/ATML/PA1-repo

print(
    "Repository:",
    REPO_ROOT,
)

print(
    "Task 3 exists:",
    TASK3_REPO_ROOT.exists(),
)

Mounted at /content/drive
/content/drive/MyDrive/ATML/PA1-repo
Repository: /content/drive/MyDrive/ATML/PA1-repo
Task 3 exists: True


In [5]:
import subprocess


print("=== Git status ===")

git_status = subprocess.run(
    [
        "git",
        "status",
        "--short",
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    check=True,
)

print(
    git_status.stdout
    if git_status.stdout
    else "Working tree clean."
)

print()
print("=== Task 3 files ===")

for path in sorted(
    TASK3_REPO_ROOT.rglob("*")
):
    if (
        path.is_file()
        and "__pycache__"
        not in path.parts
    ):
        relative_path = (
            path.relative_to(
                REPO_ROOT
            )
        )

        size_mb = (
            path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{size_mb:7.2f} MB | "
            f"{relative_path}"
        )

print()
print(
    "=== Tracked checkpoints inside task3/ ==="
)

tracked_output = subprocess.run(
    [
        "git",
        "ls-files",
        "task3",
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    check=True,
).stdout.splitlines()

tracked_checkpoints = [
    path
    for path in tracked_output
    if path.endswith(
        (
            ".pt",
            ".pth",
            ".ckpt",
        )
    )
]

print(
    "Count:",
    len(tracked_checkpoints),
)

for path in tracked_checkpoints:
    print(path)

=== Git status ===
?? task3/


=== Task 3 files ===
   0.00 MB | task3/__init__.py
   0.00 MB | task3/configs/base.yaml
   0.00 MB | task3/configs/controlled_study.yaml
   0.00 MB | task3/configs/dan_dg.yaml
   0.00 MB | task3/configs/erm.yaml
   0.00 MB | task3/configs/sam.yaml
   0.00 MB | task3/data/__init__.py
   0.00 MB | task3/data/source_loaders.py
   0.00 MB | task3/evaluation/__init__.py
   0.12 MB | task3/figures/dan_dg_controlled_study.png
   0.11 MB | task3/figures/dan_dg_training_curves.png
   0.04 MB | task3/figures/erm_source_validation.png
   0.24 MB | task3/figures/sam_training_curves.png
   0.07 MB | task3/figures/source_diagnostics.png
   0.33 MB | task3/figures/target_sketch_confusion_matrices.png
   0.56 MB | task3/figures/target_sketch_failure_examples.png
   0.08 MB | task3/figures/target_sketch_per_class.png
   0.05 MB | task3/figures/target_sketch_performance.png
   0.00 MB | task3/methods/__init__.py
   0.00 MB | task3/methods/dan_dg.py
   0.00 MB | task3/meth

In [6]:
%%writefile task3/README.md
# Task 3 — Domain Generalization

This directory contains the Task 3 domain-generalization experiments on PACS.

## Protocol

The source domains are:

- Photo
- Art Painting
- Cartoon

Sketch is reserved as the unseen target domain.

Task 3 reuses the exact source train/validation splits from Task 2 with seed 6304. All model training, checkpoint selection, controlled-study decisions, source diagnostics, and hyperparameter choices are completed without using Sketch images or labels.

Sketch labels are accessed only in the final evaluation notebook.

## Main Methods

- ERM: the Task 2 source-only checkpoint, reused without retraining.
- DAN-DG: pairwise multi-kernel MMD across the three source domains with the required main weight of 1.0.
- SAM: non-adaptive sharpness-aware minimization with radius 0.05 and AdamW as the base optimizer.

All methods use an ImageNet-pretrained ResNet-18, domain-balanced source batches, frozen batch-normalization running statistics, and source-validation checkpoint selection.

## Controlled Study

DAN-DG is evaluated with MMD weights:

- 0.1
- 1.0
- 10.0

The required main DAN-DG result remains the model trained with weight 1.0. The other values are controlled-study variants.

## Notebooks

1. `00_task3_setup.ipynb` — repository and implementation setup
2. `01_erm_baseline.ipynb` — reuse and source evaluation of ERM
3. `02_dan_dg.ipynb` — main DAN-DG experiment
4. `03_sam.ipynb` — SAM experiment
5. `04_controlled_study.ipynb` — DAN-DG alignment-weight study
6. `05_source_diagnostics.ipynb` — source-domain separability and sharpness
7. `06_final_evaluation.ipynb` — final unseen-Sketch evaluation and failure analysis

## Main Source-Validation Results

| Method | Mean source macro-F1 |
|---|---:|
| ERM | 0.9367 |
| DAN-DG, weight 1.0 | 0.0507 |
| SAM, radius 0.05 | 0.9515 |

## Final Sketch Results

| Method | Accuracy | Macro-F1 |
|---|---:|---:|
| ERM | 0.6483 | 0.6240 |
| DAN-DG, weight 1.0 | 0.0407 | 0.0112 |
| SAM, radius 0.05 | 0.6042 | 0.6301 |
| DAN-DG, weight 0.1 | 0.7147 | 0.7133 |
| DAN-DG, weight 10.0 | 0.0407 | 0.0112 |

## Reproduction Order

Run the notebooks in numerical order. Checkpoints are saved outside the Git repository under:

`MyDrive/ATML/PA1/task3/checkpoints/`

Datasets and model checkpoints are intentionally excluded from version control.

Writing task3/README.md


In [7]:
scripts_directory = (
    TASK3_REPO_ROOT
    / "scripts"
)

scripts_directory.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Created:",
    scripts_directory,
)

Created: /content/drive/MyDrive/ATML/PA1-repo/task3/scripts


In [8]:
%%writefile task3/scripts/validate_task3.py
from pathlib import Path

import json
import subprocess
import sys

import pandas as pd


REPOSITORY_ROOT = (
    Path(__file__).resolve().parents[2]
)

TASK3_ROOT = (
    REPOSITORY_ROOT / "task3"
)

REQUIRED_FILES = [
    "README.md",
    "configs/base.yaml",
    "configs/controlled_study.yaml",
    "configs/dan_dg.yaml",
    "configs/erm.yaml",
    "configs/sam.yaml",
    "data/source_loaders.py",
    "methods/erm.py",
    "methods/dan_dg.py",
    "methods/sam.py",
    "train.py",
    "notebooks/00_task3_setup.ipynb",
    "notebooks/01_erm_baseline.ipynb",
    "notebooks/02_dan_dg.ipynb",
    "notebooks/03_sam.ipynb",
    "notebooks/04_controlled_study.ipynb",
    "notebooks/05_source_diagnostics.ipynb",
    "notebooks/06_final_evaluation.ipynb",
    "results/erm_source_validation.csv",
    "results/dan_dg_source_validation.csv",
    "results/sam_source_validation.csv",
    "results/dan_dg_controlled_study_summary.csv",
    "results/source_domain_separability.csv",
    "results/source_sharpness_proxy.csv",
    "results/target_sketch_summary.csv",
    "results/target_sketch_per_class.csv",
    "results/target_sketch_predictions.csv",
    "results/target_sketch_failure_summary.csv",
    "results/final_evaluation_metadata.json",
    "figures/erm_source_validation.png",
    "figures/dan_dg_training_curves.png",
    "figures/sam_training_curves.png",
    "figures/dan_dg_controlled_study.png",
    "figures/source_diagnostics.png",
    "figures/target_sketch_performance.png",
    "figures/target_sketch_per_class.png",
    "figures/target_sketch_confusion_matrices.png",
    "figures/target_sketch_failure_examples.png",
]

missing_files = [
    relative_path
    for relative_path in REQUIRED_FILES
    if not (
        TASK3_ROOT / relative_path
    ).is_file()
]

if missing_files:
    print(
        "Missing required files:"
    )

    for relative_path in missing_files:
        print(
            " -",
            relative_path,
        )

    sys.exit(1)


target_summary = pd.read_csv(
    TASK3_ROOT
    / "results/target_sketch_summary.csv"
)

required_methods = {
    "erm",
    "dan_dg",
    "sam",
    "dan_dg_lambda_0p1",
    "dan_dg_lambda_10p0",
}

observed_methods = set(
    target_summary["method"]
)

if observed_methods != required_methods:
    raise ValueError(
        "Unexpected target-summary methods: "
        f"{sorted(observed_methods)}"
    )

if not (
    target_summary[
        "number_of_target_samples"
    ]
    == 3929
).all():
    raise ValueError(
        "Target sample count must be 3929."
    )


controlled_summary = pd.read_csv(
    TASK3_ROOT
    / "results/"
    "dan_dg_controlled_study_summary.csv"
)

observed_weights = set(
    controlled_summary[
        "mmd_weight"
    ].astype(float)
)

if observed_weights != {
    0.1,
    1.0,
    10.0,
}:
    raise ValueError(
        "Controlled-study weights are incorrect."
    )


separability = pd.read_csv(
    TASK3_ROOT
    / "results/"
    "source_domain_separability.csv"
)

sharpness = pd.read_csv(
    TASK3_ROOT
    / "results/"
    "source_sharpness_proxy.csv"
)

main_methods = {
    "erm",
    "dan_dg",
    "sam",
}

if set(
    separability["method"]
) != main_methods:
    raise ValueError(
        "Domain-separability methods are incorrect."
    )

if set(
    sharpness["method"]
) != main_methods:
    raise ValueError(
        "Sharpness methods are incorrect."
    )


metadata_path = (
    TASK3_ROOT
    / "results/"
    "final_evaluation_metadata.json"
)

with metadata_path.open(
    "r",
    encoding="utf-8",
) as metadata_file:
    metadata = json.load(
        metadata_file
    )

if metadata[
    "target_domain"
] != "sketch":
    raise ValueError(
        "Final target domain must be Sketch."
    )

if metadata[
    "number_of_target_samples"
] != 3929:
    raise ValueError(
        "Incorrect target sample count "
        "in final metadata."
    )


tracked_files = subprocess.run(
    [
        "git",
        "ls-files",
        "task3",
    ],
    cwd=REPOSITORY_ROOT,
    capture_output=True,
    text=True,
    check=True,
).stdout.splitlines()

tracked_checkpoints = [
    path
    for path in tracked_files
    if path.endswith(
        (
            ".pt",
            ".pth",
            ".ckpt",
            ".safetensors",
        )
    )
]

if tracked_checkpoints:
    raise ValueError(
        "Checkpoint files must not be tracked: "
        f"{tracked_checkpoints}"
    )


print("Task 3 validation passed.")

print(
    "Methods:",
    sorted(
        required_methods
    ),
)

print(
    "Target samples:",
    int(
        target_summary[
            "number_of_target_samples"
        ].iloc[0]
    ),
)

print(
    "Tracked checkpoints inside task3/:",
    len(tracked_checkpoints),
)

Writing task3/scripts/validate_task3.py


In [9]:
%cd /content/drive/MyDrive/ATML/PA1-repo

!python task3/scripts/validate_task3.py

print()
print("=== Python compilation ===")

!python -m compileall -q common shared task2 task3

print(
    "All Python files compiled."
)

print()
print("=== Git status ===")

!git status --short

/content/drive/MyDrive/ATML/PA1-repo
Task 3 validation passed.
Methods: ['dan_dg', 'dan_dg_lambda_0p1', 'dan_dg_lambda_10p0', 'erm', 'sam']
Target samples: 3929
Tracked checkpoints inside task3/: 0

=== Python compilation ===
All Python files compiled.

=== Git status ===
?? task3/


In [10]:
%cd /content/drive/MyDrive/ATML/PA1-repo

!git add task3

print("=== Staged-file check ===")

!git diff --cached --check

print()
print("=== Git status ===")

!git status --short

/content/drive/MyDrive/ATML/PA1-repo
=== Staged-file check ===

=== Git status ===
A  task3/README.md
A  task3/__init__.py
A  task3/configs/base.yaml
A  task3/configs/controlled_study.yaml
A  task3/configs/dan_dg.yaml
A  task3/configs/erm.yaml
A  task3/configs/sam.yaml
A  task3/data/__init__.py
A  task3/data/source_loaders.py
A  task3/evaluation/__init__.py
A  task3/figures/dan_dg_controlled_study.png
A  task3/figures/dan_dg_training_curves.png
A  task3/figures/erm_source_validation.png
A  task3/figures/sam_training_curves.png
A  task3/figures/source_diagnostics.png
A  task3/figures/target_sketch_confusion_matrices.png
A  task3/figures/target_sketch_failure_examples.png
A  task3/figures/target_sketch_per_class.png
A  task3/figures/target_sketch_performance.png
A  task3/methods/__init__.py
A  task3/methods/dan_dg.py
A  task3/methods/erm.py
A  task3/methods/sam.py
A  task3/models/__init__.py
A  task3/notebooks/00_task3_setup.ipynb
A  task3/notebooks/01_erm_baseline.ipynb
A  task3/noteboo

In [11]:
import subprocess


print("=== Staged summary ===")

!git diff --cached --stat

print()
print(
    "=== Potentially unwanted staged files ==="
)

staged_files = subprocess.run(
    [
        "git",
        "diff",
        "--cached",
        "--name-only",
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    check=True,
).stdout.splitlines()

unwanted_suffixes = (
    ".pt",
    ".pth",
    ".ckpt",
    ".safetensors",
    ".pyc",
    ".zip",
)

unwanted_files = [
    path
    for path in staged_files
    if (
        path.endswith(
            unwanted_suffixes
        )
        or "__pycache__" in path
        or "/checkpoints/" in path
    )
]

if unwanted_files:
    for path in unwanted_files:
        print(path)
else:
    print("None")

print()
print(
    "Total staged files:",
    len(staged_files),
)

=== Staged summary ===
 task3/README.md                                    |    71 +
 task3/__init__.py                                  |     0
 task3/configs/base.yaml                            |    38 +
 task3/configs/controlled_study.yaml                |    14 +
 task3/configs/dan_dg.yaml                          |    20 +
 task3/configs/erm.yaml                             |     6 +
 task3/configs/sam.yaml                             |     8 +
 task3/data/__init__.py                             |     0
 task3/data/source_loaders.py                       |   174 +
 task3/evaluation/__init__.py                       |     0
 task3/figures/dan_dg_controlled_study.png          |   Bin 0 -> 124352 bytes
 task3/figures/dan_dg_training_curves.png           |   Bin 0 -> 110493 bytes
 task3/figures/erm_source_validation.png            |   Bin 0 -> 46428 bytes
 task3/figures/sam_training_curves.png              |   Bin 0 -> 248977 bytes
 task3/figures/source_diagnostics.png               

In [12]:
%cd /content/drive/MyDrive/ATML/PA1-repo

!git commit -m "Complete Task 3 domain generalization experiments"

print()
print("=== Git status ===")

!git status

print()
print("=== Recent commits ===")

!git log --oneline --decorate -3

/content/drive/MyDrive/ATML/PA1-repo
[main 9c82ebc] Complete Task 3 domain generalization experiments
 57 files changed, 21641 insertions(+)
 create mode 100644 task3/README.md
 create mode 100644 task3/__init__.py
 create mode 100644 task3/configs/base.yaml
 create mode 100644 task3/configs/controlled_study.yaml
 create mode 100644 task3/configs/dan_dg.yaml
 create mode 100644 task3/configs/erm.yaml
 create mode 100644 task3/configs/sam.yaml
 create mode 100644 task3/data/__init__.py
 create mode 100644 task3/data/source_loaders.py
 create mode 100644 task3/evaluation/__init__.py
 create mode 100644 task3/figures/dan_dg_controlled_study.png
 create mode 100644 task3/figures/dan_dg_training_curves.png
 create mode 100644 task3/figures/erm_source_validation.png
 create mode 100644 task3/figures/sam_training_curves.png
 create mode 100644 task3/figures/source_diagnostics.png
 create mode 100644 task3/figures/target_sketch_confusion_matrices.png
 create mode 100644 task3/figures/target_sk